In [ ]:
from pathlib import Path
import sys
PROJETO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJETO / "scripts"))


# Renda do responsável por setor censitário — Brasil, Censo 2022

Objetivo: uma linha por setor da **malha definitiva nacional de 2022**, sem visualizações.
Os dados de renda são os agregados do universo atualizados em **08/05/2026**.

## Avaliação das fontes e das malhas

- O GeoPandas lê shapefiles e GeoPackages, mas não inclui a malha dos setores do Brasil.
- O IBGE fornece a malha nacional e por UF. Não é necessário baixar os arquivos manualmente.
- Para esta tabela, precisamos apenas do DBF (atributos) da malha nacional, sem os polígonos.
  O script extrai esse componente por intervalos HTTP, evitando baixar o ZIP completo (~774 MB).
- A malha é a base do `left join`: setores ausentes no arquivo de renda permanecem na saída.
- A chave `CD_SETOR` tem 15 dígitos e deve ser tratada como texto, inclusive na aplicação.
- Não se deve misturar esta tabela com a malha preliminar de 2022 ou com malhas de outros anos.

Fontes: [renda e dicionário](https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios_Rendimento_do_Responsavel/)
e [malha definitiva](https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/setores/shp/BR/).


## Preparação

Instale as dependências no ambiente do kernel, executando `python -m pip install -r requirements.txt` no terminal, na raiz do projeto.
A primeira célula configura os imports tanto na raiz quanto na pasta `notebooks`.
Os arquivos locais em `dados/brutos` são reutilizados. O primeiro uso exige acesso ao IBGE.


In [ ]:
from gerar_tabela import gerar, OUT

renda_setor_df = gerar()


## Dicionário das variáveis

As variáveis oficiais são preservadas para facilitar conferência com o IBGE:

| Coluna | Significado |
| --- | --- |
| V06001 | Pessoas responsáveis em domicílios particulares permanentes ocupados |
| V06002 | Moradores nesses domicílios |
| V06003 | Variância do número de moradores nesses domicílios |
| V06004 | Rendimento nominal médio mensal dos responsáveis **com rendimentos**, em R$ |
| V06005 | Variância do rendimento nominal mensal dos responsáveis com rendimentos, em R$² |
| V06006 | Rendimento nominal mediano mensal dos responsáveis com rendimentos, em R$ |

As três últimas variáveis referem-se aos responsáveis por domicílios particulares permanentes ocupados.
São valores nominais do Censo 2022, sem atualização monetária. **Não representam renda domiciliar total nem renda per capita.**

Cada variável tem uma coluna `*_simbolo` que preserva os símbolos originais (`X`, `.`, etc.).
O valor numérico fica nulo nesses casos, nunca convertido em zero. `X` indica supressão;
`.` é mantido como valor sem publicação numérica, sem inferir renda zero.
`presente_arquivo_renda` distingue setores que constam da fonte de renda dos incluídos apenas pela malha.
`status_renda_media` descreve a disponibilidade de **V06004**, não a de todas as variáveis.


In [ ]:
# Conferência tabular; sem gráficos ou mapas.
resumo = {
    "setores": len(renda_setor_df),
    "ufs": renda_setor_df["CD_UF"].nunique(),
    "setores_com_registro_de_renda": int(renda_setor_df["presente_arquivo_renda"].sum()),
    "status_renda_media": renda_setor_df["status_renda_media"].value_counts().to_dict(),
}
resumo


In [ ]:
renda_setor_df.head()


## Saída para a aplicação

`dados/processados/renda_responsavel_setor_2022_BR.csv`: CSV UTF-8, separado por vírgulas,
com ponto decimal e campos vazios para valores numéricos ausentes.
`dados/processados/validacao.json`: cobertura, contagens, fontes e hashes dos arquivos.

Para associar futuramente às geometrias, use `CD_SETOR` com a malha definitiva de 2022,
validando a associação como um para um. Esta etapa não gera nem exporta geometrias.


In [ ]:
# Exemplo de leitura do arquivo entregue; códigos continuam sendo texto.
import pandas as pd

codigos = ["CD_SETOR", "CD_UF", "CD_MUN", "CD_SIT", "CD_TIPO"]
tabela_aplicacao = pd.read_csv(
    OUT / "renda_responsavel_setor_2022_BR.csv",
    dtype={col: "string" for col in codigos},
)
assert tabela_aplicacao["CD_SETOR"].is_unique
assert len(tabela_aplicacao) == len(renda_setor_df)
